In [ ]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

TypeError: int cp_fit_final(TChain& chain, TString mode, bool check_corr, Controls::DataType& data_type, ErrorHandling::ErrorLogs& logger, KLOE::pm00& Obj, ConfigWatcher& cfgWatcher) =>
    TypeError: takes at least 7 arguments (0 given)

In [3]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)

data_files = []

for i in range(1, 46, 1):
    data_files += glob.glob(
        f"/workspace/root_files/hal/DATA_SEMILEPTONIC_NoSmearing/dk0*_{i}.root"
    )

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

# Scaling factor
lumi_data = 183723.1100
lumi_mc = 557534.7841
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    ###############################################################################################
    .Filter("mcflag == 0 && mctruth != -1")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01", f"pi01[5] - {mPi0}")
    .Define("MassPi02", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    ###############################################################################################    
    .Filter("mcflag == 1 && mctruth != -1")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 1"
)

df_Background = df.Filter(
    f"mcflag == 1 && mctruth != 1 && mctruth != -1"
)

Przetestowano 135 plików. Poprawnych: 135, Odrzuconych: 0
Przetestowano 45 plików. Poprawnych: 45, Odrzuconych: 0
Lumi factor: 0.3295


In [4]:
current_date = time.strftime("%Y-%m-%d")
current_result_dir = f"results/signal_separated_kin_fits/{current_date}"

metadata = {
    "saved_errors": "Semileptonic",
    "cutset": "NONE",
    "luminosity_pb_MC": lumi_mc,
    "luminosity_pb_data": lumi_data,
    "scale_factor": scaling_factor,
    "timestamp": current_date,
}

os.makedirs(current_result_dir, exist_ok=True)
with open(f"{current_result_dir}/analysis_config.json", "w") as f:
  json.dump(metadata, f, indent=4)

In [5]:
# Generate overlapped histograms for all channels
variables_1D = ["Knerec_5", 
                "KnerecSix_5", 
                "Kchrec_5",
                "Chi2SignalKinFit",
                "Chi2NeutralKinFit",
                "Chi2PMKinFit",
                "MassPi01Fit",
                "MassPi02Fit",
                "MassPi0NeutralFit1",
                "MassPi0NeutralFit2",
                "MassPi0PMFit1",
                "MassPi0PMFit2",
                "MassPi01",
                "MassPi02",
                "Delta_t_fit",
                "Delta_t_MC",
                "Delta_t_rec",
                "Delta_t_fit_resolution", "Delta_t_rec_resolution",
                "trk1_px_resolution", "trk2_px_resolution", "trk1_py_resolution", "trk2_py_resolution", "trk1_pz_resolution", "trk2_pz_resolution"
                ]
variables_1D_titles = ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]",
                       "#chi^{2}_{+-00} [-]",
                       "#chi^{2}_{00} [-]",
                       "#chi^{2}_{+-} [-]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "#Deltat^{fit} [#tau_{S}]",
                       "#Deltat^{MC} [#tau_{S}]",
                       "#Deltat^{rec} [#tau_{S}]",
                       "#Deltat^{fit} - #Deltat^{MC} [#tau_{S}]",
                       "#Deltat^{rec} - #Deltat^{MC} [#tau_{S}]",
                       "#p^{x,rec}_{+-,1} - p^{x,MC}_{+-,1} [MeV/c]",
                       "#p^{x,rec}_{+-,2} - p^{x,MC}_{+-,2} [MeV/c]",
                       "#p^{y,rec}_{+-,1} - p^{y,MC}_{+-,1} [MeV/c]",
                       "#p^{y,rec}_{+-,2} - p^{y,MC}_{+-,2} [MeV/c]",
                       "#p^{z,rec}_{+-,1} - p^{z,MC}_{+-,1} [MeV/c]",
                       "#p^{z,rec}_{+-,2} - p^{z,MC}_{+-,2} [MeV/c]"
                       ]
range_X_1D = [(100, -400, 200), 
              (100, -400, 200), 
              (100, -200, 200),
              (100, 0, 500),
              (100, 0, 500),
              (100, 0, 100),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (100, -400, 400),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300),
              (401, -300, 300)
              ]

# Zmienne widoczne TYLKO dla jednego, wybranego mctruth (np. prawdziwe wartości z MC,
# dostępne tylko w próbce sygnałowej) - bez Data i bez MC sum. Można wskazać
# dowolne mctruth (niekoniecznie 1) ustawiając TRUTH_MCTRUTH_ID_1D.
TRUTH_ONLY_VARS_1D = {"Delta_t_MC", "Delta_t_fit_resolution", "Delta_t_rec_resolution", "trk1_px_resolution", "trk2_px_resolution", "trk1_py_resolution", "trk2_py_resolution", "trk1_pz_resolution", "trk2_pz_resolution"}
TRUTH_MCTRUTH_ID_1D = 1

In [13]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = [
            "abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
            "Chi2SignalKinFit < 30.0"
            ]

cuts_json_str = json.dumps(cut_list, indent=2)
# Pusta lista cięć (np. do podglądu histogramów "przed cięciami") dostaje
# własny, czytelny tag zamiast hasha liczonego z "[]" (co zawsze dawałoby
# ten sam hash i nadpisywało poprzednie wyniki bez cięć).
cut_hash = "nocuts" if not cut_list else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]

In [14]:
# 1D histograms for all_channels
modes = {"full": [], "zoom": []}

# Define histogram ranges for different modes
modes["full"] = [(100, -400, 200), 
                 (100, -400, 200), 
                 (100, -200, 200),
                 (100, 0, 500),
                 (100, 0, 500),
                 (100, 0, 100),
                 (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300)
                  ]
modes["zoom"] = [(100, -100, 100), 
                (100, -100, 100), 
                (100, -5, 5),
                (100, 0, 100),
                (100, 0, 100),
                (100, 0, 30),
                (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (133, -100, 100),
                              (133, -100, 100),
                              (133, -100, 100),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10)
                              ]

total_cut = build_cut_expression(cut_list)

(vars_1D_main, titles_1D_main, modes_1D_main), (vars_1D_truth, titles_1D_truth, modes_1D_truth) = \
    split_variable_group(variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D)

hist_1D = fill_1d_histograms(
    df, df_data, channName, vars_1D_main, titles_1D_main, modes_1D_main,
    total_cut, scaling_factor, compute_mc_sum=True
)

hist_1D_truth = {}
if vars_1D_truth:
  truth_channels_1D = {TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]}
  hist_1D_truth = fill_1d_histograms(
      df, df_data, truth_channels_1D, vars_1D_truth, titles_1D_truth, modes_1D_truth,
      total_cut, compute_mc_sum=False
  )

In [16]:
root_file_name = "separated_kin_fits_1D_variables.root"
hist_1D, canvas, legends = draw_and_save_1d_histograms(
    hist_1D, vars_1D_main, modes_1D_main, channColor, channTitle,
    current_result_dir, cut_hash, cuts_json_str, cut_list, root_file_name
)

if vars_1D_truth:
  root_file_name_truth = "separated_kin_fits_1D_truth_variables.root"
  hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
      hist_1D_truth, vars_1D_truth, modes_1D_truth, channColor, channTitle,
      current_result_dir, cut_hash, cuts_json_str, cut_list, root_file_name_truth
  )

for canva in canvas:
  canva.Close()
if vars_1D_truth:
  for canva in canvas_truth:
    canva.Close()

[INFO] Zaktualizowano wpis dla hashu 17e4f29b w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 17e4f29b w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv


In [19]:
# Zdefiniuj swoje pełne listy cięć i parametry tak jak dotychczas:
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

cut1 = 8.086
cut2 = 11.515

full_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
    "Chi2SignalKinFit < 30.0",
]

# 1D histograms for all_channels
modes = {"full": [], "zoom": []}

modes["full"] = [(100, -400, 200), 
                 (100, -400, 200), 
                 (100, -200, 200),
                 (100, 0, 500),
                 (100, 0, 500),
                 (100, 0, 100),
                 (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (100, -400, 400),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300),
                  (401, -300, 300)
                  ]
modes["zoom"] = [(100, -100, 100), 
                (100, -100, 100), 
                (100, -5, 5),
                (100, 0, 100),
                (100, 0, 100),
                (100, 0, 30),
                (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (100, -150, 150),
                              (133, -100, 100),
                              (133, -100, 100),
                              (133, -100, 100),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10),
                              (133, -10, 10)
                              ]

# --- GŁÓWNA PĘTLA BUDUJĄCA CIĘCIA KROK PO KROKU ---
# Możesz zacząć od indeksu -1 (czyli stan "beż żadnych cięć" / nocuts)
# lub od i = 0 (jeśli chcesz zacząć od razu od pierwszego cięcia).
for i in range(-1, len(full_cut_list)):

  # Wycinamy podzbiór cięć od początku do bieżącego kroku
  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  print(
      f"\n[INFO] Przetwarzanie kroku: {step_name} (liczba aktywnych cięć:"
      f" {len(current_cut_list)})"
  )

  # Przygotowanie metadanych, hashu i unikalnego katalogu dla tego kroku
  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
      "nocuts"
      if not current_cut_list
      else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
  )

  # Możesz opcjonalnie rozbić wyniki na podkatalogi dla każdego kroku, np.:
  # step_result_dir = os.path.join(current_result_dir, step_name)
  step_result_dir = (
      current_result_dir  # Lub zapisywać w jednym miejscu z hashem w nazwie pliku
  )

  # Składanie wyrażenia cięć za pomocą Twojej funkcji
  total_cut = build_cut_expression(current_cut_list)

  # Podział zmiennych
  (vars_1D_main, titles_1D_main, modes_1D_main), (
      vars_1D_truth,
      titles_1D_truth,
      modes_1D_truth,
  ) = split_variable_group(
      variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D
  )

  # Wypełnianie histogramów głównych
  hist_1D = fill_1d_histograms(
      df,
      df_data,
      channName,
      vars_1D_main,
      titles_1D_main,
      modes_1D_main,
      total_cut,
      scaling_factor,
      compute_mc_sum=True,
  )

  # Wypełnianie histogramów truth (jeśli istnieją)
  hist_1D_truth = {}
  if vars_1D_truth:
    truth_channels_1D = {TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]}
    hist_1D_truth = fill_1d_histograms(
        df,
        df_data,
        truth_channels_1D,
        vars_1D_truth,
        titles_1D_truth,
        modes_1D_truth,
        total_cut,
        compute_mc_sum=False,
    )

  # Rysowanie i zapis do dedykowanych plików ROOT / PDF
  # (zakładam, że Twoja funkcja draw_and_save przyjmuje cut_hash i zapisuje pliki)
  root_file_name = f"separated_kin_fits_1D_variables.root"
  hist_1D, canvas, legends = draw_and_save_1d_histograms(
      hist_1D,
      vars_1D_main,
      modes_1D_main,
      channColor,
      channTitle,
      step_result_dir,
      cut_hash,
      cuts_json_str,
      current_cut_list,
      root_file_name,
  )

  if vars_1D_truth:
    root_file_name_truth = (
        f"separated_kin_fits_1D_truth_variables.root"
    )
    hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
        hist_1D_truth,
        vars_1D_truth,
        modes_1D_truth,
        channColor,
        channTitle,
        step_result_dir,
        cut_hash,
        cuts_json_str,
        current_cut_list,
        root_file_name_truth,
    )

  # Sprzątanie obiektów Canvas z pamięci ROOT-a
  for canva in canvas:
    canva.Close()
  if vars_1D_truth:
    for canva in canvas_truth:
      canva.Close()

print(
    "[INFO] Pętla sekwencyjnego nakładania cięć została pomyślnie zakończona!"
)


[INFO] Przetwarzanie kroku: 00_no_cuts (liczba aktywnych cięć: 0)
[INFO] Zaktualizowano wpis dla hashu nocuts w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu nocuts w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_01 (liczba aktywnych cięć: 1)
[INFO] Zaktualizowano wpis dla hashu 26f97643 w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 26f97643 w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_02 (liczba aktywnych cięć: 2)
[INFO] Zaktualizowano wpis dla hashu 17dde94b w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 17dde94b w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_03 (liczba ak

### Get 2D histograms for each cut

In [26]:
variables_2D = [["Knerec_5", "KnerecSix_5"], 
                ["Chi2SignalKinFit", "Chi2NeutralKinFit"], 
                ["Chi2SignalKinFit", "Chi2PMKinFit"],
                ["Chi2PMKinFit", "Chi2NeutralKinFit"],
                ["MassPi01Fit", "MassPi02Fit"],
                ["MassPi01", "MassPi02"],
                ["MassPi0NeutralFit1", "MassPi0NeutralFit2"],
                ["MassPi0PMFit1", "MassPi0PMFit2"],
                ["Delta_t_MC", "Delta_t_fit_resolution"],
                ["Delta_t_MC", "Delta_t_rec_resolution"]]
variables_2D_titles = [
                       ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{00} [-]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{+-} [-]"],
                       ["#chi^{2}_{+-} [-]", "#chi^{2}_{00} [-]"],
                       ["m^{inv,+-00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,+-}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{fit} - #Delta t^{MC} [#tau_{S}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{rec} - #Delta t^{MC} [#tau_{S}]"]
                       ]
range_X_2D = [[(100, -400, 200), (100, -400, 200)], 
              [(100, 0, 100), (100, 0, 100)], 
              [(100, 0, 100), (100, 0, 30)],
              [(100, 0, 30), (100, 0, 100)], 
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -300, 300), (100, -20, 20)],
              [(100, -300, 300), (100, -20, 20)]]

# Zmienne 2D widoczne TYLKO dla jednego, wybranego mctruth - analogicznie do 1D.
# Przynależność do grupy "truth-only" sprawdzana jest po kluczu "x_y" (var_key).
TRUTH_ONLY_VARS_2D = set([("Delta_t_MC", "Delta_t_fit_resolution"), ("Delta_t_MC", "Delta_t_rec_resolution")])
TRUTH_MCTRUTH_ID_2D = 1

In [27]:
## 2D histograms
hist_2D = {}
modes = {}

# Definicja zakresów dla trybów: (nbinsX, xlow, xhigh, nbinsY, ylow, yhigh)
modes["full"] = [(100, -400, 200, 100, -400, 200), 
                 (100, 0, 100, 100, 0, 100), 
                 (100, 0, 100, 100, 0, 30),
                 (100, 0, 30, 100, 0, 100),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -300, 300, 100, -20, 20),
                 (100, -300, 300, 100, -20, 20)
                 ]
modes["zoom"] = [(100, -100, 100, 100, -100, 100), 
                 (100, 0, 50, 100, 0, 50), 
                 (100, 0, 50, 100, 0, 15),
                  (100, 0, 15, 100, 0, 50),
                 (100, -50, 50, 100, -50, 50),
                 (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -100, 100, 100, -10, 10),
                (100, -100, 100, 100, -10, 10)]

(vars_2D_main, titles_2D_main, modes_2D_main), (vars_2D_truth, titles_2D_truth, modes_2D_truth) = \
    split_variable_group(
        variables_2D, variables_2D_titles, modes,
        is_truth_only=lambda pair: "_".join(pair) in TRUTH_ONLY_VARS_2D
    )

hist_2D = fill_2d_histograms(
    df, df_data, channName, vars_2D_main, titles_2D_main, modes_2D_main,
    total_cut, scaling_factor, compute_mc_sum=True
)

hist_2D_truth = {}
if vars_2D_truth:
  truth_channels_2D = {TRUTH_MCTRUTH_ID_2D: channName[TRUTH_MCTRUTH_ID_2D]}
  hist_2D_truth = fill_2d_histograms(
      df, df_data, truth_channels_2D, vars_2D_truth, titles_2D_truth, modes_2D_truth,
      total_cut, compute_mc_sum=False
  )

In [28]:
root_file_name = "separated_kin_fits_2D_variables.root"
hist_2D, canva = draw_and_save_2d_histograms(
    hist_2D, vars_2D_main, modes_2D_main, current_result_dir, cut_hash,
    cuts_json_str, cut_list, root_file_name
)

if vars_2D_truth:
  root_file_name_truth = "separated_kin_fits_2D_truth_variables.root"
  hist_2D_truth, canva_truth = draw_and_save_2d_histograms(
      hist_2D_truth, vars_2D_truth, modes_2D_truth, current_result_dir, cut_hash,
      cuts_json_str, cut_list, root_file_name_truth
  )

for canva in canvas:
  canva.Close()
if vars_1D_truth:
  for canva in canvas_truth:
    canva.Close()

[INFO] Zaktualizowano wpis dla hashu 17e4f29b w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv


In [4]:
import pandas as pd
import json
import hashlib

df = pd.read_csv("results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv")

hash_dict = df[["hash", "cuts_description"]]
hash_dict = hash_dict.set_index("hash").to_dict()["cuts_description"]

# Zdefiniuj swoje pełne listy cięć i parametry tak jak dotychczas:
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

cut1 = 8.086
cut2 = 11.515

# Hash list order
full_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
    "Chi2SignalKinFit < 30.0",
]

hash_order = []
for i in range(-1, len(full_cut_list)):

  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
        "nocuts"
        if not current_cut_list
        else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
    )

  hash_order.append(cut_hash)

In [6]:
import json
import ROOT

# Getting purity, efficiency and number of events for each cut
results_file = ROOT.TFile("results/signal_separated_kin_fits/2026-10-22/separated_kin_fits_1D_variables.root", "READ")

no_cuts_hash = "nocuts"
num_of_events = {}
purity = {}

for hash, cut in hash_dict.items():
    obj_integrals = results_file.Get(f"{hash}/ChannelIntegrals")
    obj_purity = results_file.Get(f"{hash}/Purity")
    if obj_integrals:
      if hasattr(obj_integrals, "GetTitle"):
        num_of_events[hash] = json.loads(obj_integrals.GetTitle())

    if obj_purity:
      if hasattr(obj_purity, "GetVal"):
        purity[hash] = obj_purity.GetVal()

In [7]:
import numpy as np
import ROOT

x, y = [], []

# Pamiętaj o użyciu innej nazwy zmiennej niż 'hash' (wbudowana funkcja Pythona)
for i, curr_hash in enumerate(hash_order):
  x.append(float(i))
  y.append(float(purity[curr_hash]))

canva = ROOT.TCanvas("canva", "Purity vs Cut Step", 800, 600)
canva.SetLogy(True)  # Skala logarytmiczna
ROOT.gPad.SetGrid(True, True)

# Liczba binów musi odpowiadać liczbie elementów w y
n_bins = len(y)
h_purity = ROOT.TH1D(
    "h_purity", "Czystość w kolejnych krokach cięć;Krok selekcji;Czystość", n_bins, 0, n_bins
)

# Przykładowe etykiety dla osi X (dostosuj do liczby kroków w hash_order)
labels = ["No cuts", "Cut 1", "Cut 2", "Cut 3", "Cut 4", "Cut 5", "Cut 6"]

for i, val in enumerate(y):
  h_purity.SetBinContent(i + 1, val)
  # Dodanie czytelnych opisów do każdego binu (indeksowanie w ROOT od 1)
  if i < len(labels):
    h_purity.GetXaxis().SetBinLabel(i + 1, labels[i])

h_purity.SetStats(0)  # Usuwa stat box
h_purity.SetFillColor(ROOT.kAzure + 4)

h_purity.Draw("HIST")

# Obrócenie etykiet osi X, jeśli nazwy cięć są dłuższe
h_purity.GetXaxis().SetLabelSize(0.035)
canva.SetBottomMargin(0.2)  # Zostawienie miejsca na dole na opisy

canva.Draw()

### Test of cuts for separated kinematic fit

In [6]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {cut1}, {cut2})",
            "Chi2NeutralKinFit < 30.0 && Chi2PMKinFit < 30.0"
            ]

In [7]:
# Wypuszczasz dane po cięciach bazowych do lokalnego pliku raz przed pętlą:
# 1. Bierzemy wszystkie cięcia OPRÓCZ ostatnich dwóch (do optymalizacji)
base_cuts = cut_list[:-2]
print("Cięcia bazowe do cache:", base_cuts)

# 2. Łączymy je w jeden string C++ za pomocą ' && '
total_base_cut = " && ".join([f"({cut})" for cut in base_cuts])

columns_to_save = [
    # Zdefiniowane zmienne korelacji mas
    "Kchrec_5",
    "Knerec_5",
    "KnerecSix_5",
    "KnetriKinFit_5",
    "MassPi01",
    "MassPi02",
    "MassPi01Fit",
    "MassPi02Fit",
    "MassPi0NeutralFit1",
    "MassPi0NeutralFit2",
    "MassPi0PMFit1",
    "MassPi0PMFit2",
    
    # Cylindryczne i współrzędne
    "rho_ch",
    "rho_ne",
    "z_ch",
    "z_ne",
    
    # Omega-pi0 geometrical cut variables
    "bhabha_IP",
    "bhabha_IP_changed",
    "rho_00_closest",
    "rho_pm_closest",
    "rho_omega",
    "z_00_closest",
    "z_pm_closest",
    
    # Bad tracks geometrical cut variables
    "rho_pm_00",
    "z_pm_00",
    "cos_trk1_K",
    "cos_trk2_K",
    
    # Delta t
    "Delta_t_fit",
    "Delta_t_MC",
    "Delta_t_rec",
    
    # Resolutions i matching
    "Delta_t_fit_resolution",
    "Delta_t_rec_resolution",
    "trk1_matched",
    "trk2_matched",
    "trk1_px_resolution",
    "trk2_px_resolution",
    "trk1_py_resolution",
    "trk2_py_resolution",
    "trk1_pz_resolution",
    "trk2_pz_resolution",

    # Mctruth
    "mctruth",

    # Chi2
    "Chi2SignalKinFit",
    "Chi2NeutralKinFit",
    "Chi2PMKinFit"
]

# Konwersja listy Pythona na wektor C++ wymagany przez RDataFrame.Snapshot w PyROOT
col_vec = ROOT.std.vector("string")()
for col in columns_to_save:
    col_vec.push_back(col)

# 3. Filtrujemy i robimy Snapshot
df_base = df.Filter(total_base_cut)
df_base.Snapshot(
    "tree_name", "/workspace/root_files/hal/local_root_cache.root", col_vec
)

df_base = df_data.Filter(total_base_cut)
df_base.Snapshot(
    "tree_name", "/workspace/root_files/hal/local_root_cache_data.root", col_vec
)

Cięcia bazowe do cache: ['abs(Kchrec_5) <= 3 * 0.8090', 'ellipse_cut_func(Knerec_5, KnerecSix_5)', '(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0)', '(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45 && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45)']


<cppyy.gbl.ROOT.RDF.RResultPtr<ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager,void> > object at 0x55981186eb80>

In [8]:
import hashlib
import json
import numpy as np
import ROOT
from scipy.optimize import minimize

# --- 1. DEFINICJA STAŁYCH CIĘĆ BAZOWYCH ---
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

base_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
]

# 1D histograms modes (nienaruszone)
modes = {"full": [], "zoom": []}

modes["full"] = [
    (100, -400, 200),
    (100, -400, 200),
    (100, -200, 200),
    (100, 0, 500),
    (100, 0, 500),
    (100, 0, 100),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
]
modes["zoom"] = [
    (100, -100, 100),
    (100, -100, 100),
    (100, -5, 5),
    (100, 0, 100),
    (100, 0, 100),
    (100, 0, 30),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (133, -100, 100),
    (133, -100, 100),
    (133, -100, 100),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
]

# --- 1.1 DOCELOWE WARTOŚCI Z PIERWOTNEJ ANALIZY ---
TARGET_PURITY = 0.22  # Zastąp pożądaną czystością
TARGET_EFFICIENCY = 0.5593  # Zastąp pożądaną wydajnością

# Zmienna globalna do przechowywania początkowej liczby zliczeń sygnału (baza bez cięcia m1_vs_m2)
INITIAL_SIGNAL_YIELD = None

# Zmienna globalna do zliczania iteracji w algorytmie
iteration_counter = 0


# --- 2. FUNKCJA POMOCNICZA DO POBRANIA SYGNAŁU Z PLIKU ROOT ---
def extract_signal_from_file(file_path, hash):
  f_res = ROOT.TFile(file_path, "READ")
  obj_integrals = f_res.Get(f"{hash}/ChannelIntegrals")
  signal_val = 0.0
  if obj_integrals and hasattr(obj_integrals, "GetTitle"):
    title_str = obj_integrals.GetTitle()
    try:
      # TTitle przechowuje słownik w formacie JSON (widocznym na załączionym zrzucie ekranu)[cite: 1]
      import json as pyjson

      data_dict = pyjson.loads(title_str)
      signal_val = float(data_dict.get("Signal", 0.0))
    except Exception as e:
      print(f"[OSTRZEŻENIE] Nie udało się sparsować ChannelIntegrals: {e}")
  f_res.Close()
  return signal_val


# --- 3. KROK POCZĄTKOWY (ITERACJA 0) - BAZOWY DLA WYDAJNOŚCI ---
print(
    "[INFO] Wykonywanie kroku początkowego (iteracja 0) bez cięcia m1_vs_m2..."
)
df_init_mc = ROOT.RDataFrame(
    "tree_name", "/workspace/root_files/hal/local_root_cache.root"
)
df_init_data = ROOT.RDataFrame(
    "tree_name", "/workspace/root_files/hal/local_root_cache_data.root"
)

init_cut_list = list(base_cut_list)  # Tylko cięcia bazowe
init_cuts_json_str = json.dumps(init_cut_list, indent=2)
init_cut_hash = hashlib.md5(init_cuts_json_str.encode("utf-8")).hexdigest()[:8]
init_total_cut = (
    "true"  # Brak dodatkowego cięcia, bazowe są już w plikach Snapshot
)

(vars_1D_main, titles_1D_main, modes_1D_main), (
    vars_1D_truth,
    titles_1D_truth,
    modes_1D_truth,
) = split_variable_group(
    variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D
)

hist_1D_init = fill_1d_histograms(
    df_init_mc,
    df_init_data,
    channName,
    vars_1D_main,
    titles_1D_main,
    modes_1D_main,
    init_total_cut,
    scaling_factor,
    compute_mc_sum=True,
)

root_file_name_init = "separated_kin_fits_optimization_1D_variables.root"
_, canvas_init, _ = draw_and_save_1d_histograms(
    hist_1D_init,
    vars_1D_main,
    modes_1D_main,
    channColor,
    channTitle,
    current_result_dir,
    init_cut_hash,
    init_cuts_json_str,
    init_cut_list,
    root_file_name_init,
)
for c in canvas_init:
  c.Close()

init_file_path = f"{current_result_dir}/{root_file_name_init}"
INITIAL_SIGNAL_YIELD = extract_signal_from_file(init_file_path, init_cut_hash)
print(
    "[INFO] Początkowa liczba sygnału (Initial Signal Yield):"
    f" {INITIAL_SIGNAL_YIELD:.2f}"
)


# --- 4. FUNKCJA CELU DLA NELDER-MEAD ---
def objective_function(params):
  global iteration_counter, INITIAL_SIGNAL_YIELD
  iteration_counter += 1
  cut1, cut2 = params

  df_current = ROOT.RDataFrame(
      "tree_name", "/workspace/root_files/hal/local_root_cache.root"
  )
  df_data_current = ROOT.RDataFrame(
      "tree_name", "/workspace/root_files/hal/local_root_cache_data.root"
  )

  current_cut_list = list(base_cut_list)
  final_cut_string = (
      f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})"
  )
  current_cut_list.append(final_cut_string)

  print(
      f"\n[INFO] --- Nelder-Mead Iteracja {iteration_counter} --- | cut1:"
      f" {cut1:.4f}, cut2 = {cut2:.4f}"
  )

  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
  step_result_dir = current_result_dir
  total_cut = final_cut_string

  (vars_1D_main, titles_1D_main, modes_1D_main), (
      vars_1D_truth,
      titles_1D_truth,
      modes_1D_truth,
  ) = split_variable_group(
      variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D
  )

  hist_1D = fill_1d_histograms(
      df_current,
      df_data_current,
      channName,
      vars_1D_main,
      titles_1D_main,
      modes_1D_main,
      total_cut,
      scaling_factor,
      compute_mc_sum=True,
  )

  hist_1D_truth = {}
  if vars_1D_truth:
    truth_channels_1D = {TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]}
    hist_1D_truth = fill_1d_histograms(
        df_current,
        None,
        truth_channels_1D,
        vars_1D_truth,
        titles_1D_truth,
        modes_1D_truth,
        total_cut,
        compute_mc_sum=False,
    )

  root_file_name = f"separated_kin_fits_optimization_1D_variables.root"
  hist_1D, canvas, legends = draw_and_save_1d_histograms(
      hist_1D,
      vars_1D_main,
      modes_1D_main,
      channColor,
      channTitle,
      step_result_dir,
      cut_hash,
      cuts_json_str,
      current_cut_list,
      root_file_name,
  )

  if vars_1D_truth:
    root_file_name_truth = (
        f"separated_kin_fits_optimization_1D_truth_variables.root"
    )
    hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
        hist_1D_truth,
        vars_1D_truth,
        modes_1D_truth,
        channColor,
        channTitle,
        step_result_dir,
        cut_hash,
        cuts_json_str,
        current_cut_list,
        root_file_name_truth,
    )

  for canva in canvas:
    canva.Close()
  if vars_1D_truth:
    for canva in canvas_truth:
      canva.Close()

  # --- POBRANIE WYNIKU CZYSTOŚCI Z ROOT-a ---
  file_path = f"{step_result_dir}/{root_file_name}"
  f_res = ROOT.TFile(file_path, "READ")
  obj_purity = f_res.Get(f"{cut_hash}/Purity")

  purity_val = 0.0
  if obj_purity and hasattr(obj_purity, "GetVal"):
    purity_val = obj_purity.GetVal()

  # --- POBRANIE BIEŻĄCEGO SYGNAŁU I OBLICZENIE WYDAJNOŚCI ---
  current_signal = extract_signal_from_file(file_path, cut_hash)
  f_res.Close()

  current_efficiency = (
      (current_signal / INITIAL_SIGNAL_YIELD)
      if INITIAL_SIGNAL_YIELD and INITIAL_SIGNAL_YIELD > 0
      else 0.0
  )

  # --- FUNKCJA KOSZTU (LOSS) DLA DOPASOWANIA DO CELU ---
  weight_purity = 1.0
  weight_eff = 1.0

  diff_purity = purity_val - TARGET_PURITY
  diff_eff = current_efficiency - TARGET_EFFICIENCY

  loss = (weight_purity * (diff_purity ** 2)) + (
      weight_eff * (diff_eff ** 2)
  )

  print(
      f"[INFO] Czystość: {purity_val:.4f} (Cel: {TARGET_PURITY}) | Wydajność:"
      f" {current_efficiency:.4f} (Cel: {TARGET_EFFICIENCY}) | Loss:"
      f" {loss:.6f}"
  )

  return loss


# --- 5. URUCHOMIENIE OPTYMALIZACJI ---
initial_guess = [8.086, 11.515]

print(
    "[INFO] Rozpoczynanie optymalizacji metodą Nelder-Mead pod kątem zadanej"
    " czystości i wydajności..."
)

result = minimize(
    objective_function,
    initial_guess,
    method="Nelder-Mead",
    options={
        "maxiter": 30,
        "disp": True,
        "xatol": 0.05,
        "fatol": 0.0001,
    },
)

print("\n[INFO] Optymalizacja zakończona!")
print(f"Status sukcesu: {result.success}")
print(f"Komunikat: {result.message}")
print(f"Optymalne cut1: {result.x[0]:.4f}")
print(f"Optymalne cut2: {result.x[1]:.4f}")
print(f"Optymalne cuty: {result.x}")
print(f"Optymalna wartość funkcji celu (loss): {result.fun:.6f}")

[INFO] Wykonywanie kroku początkowego (iteracja 0) bez cięcia m1_vs_m2...
[INFO] Zaktualizowano wpis dla hashu 5fc561ef w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Początkowa liczba sygnału (Initial Signal Yield): 35570.53
[INFO] Rozpoczynanie optymalizacji metodą Nelder-Mead pod kątem zadanej czystości i wydajności...

[INFO] --- Nelder-Mead Iteracja 1 --- | cut1: 8.0860, cut2 = 11.5150
[INFO] Zaktualizowano wpis dla hashu be79154d w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu be79154d w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Czystość: 0.2218 (Cel: 0.22) | Wydajność: 0.5593 (Cel: 0.5593) | Loss: 0.000003

[INFO] --- Nelder-Mead Iteracja 2 --- | cut1: 8.4903, cut2 = 11.5150
[INFO] Zaktualizowano wpis dla hashu 3d4b2f27 w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis 

KeyboardInterrupt: 

In [ ]:
import hashlib
import json
import numpy as np
import ROOT
from scipy.optimize import minimize

# --- 1. DEFINICJA STAŁYCH CIĘĆ BAZOWYCH ---
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

base_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
]

# 1D histograms modes (nienaruszone)
modes = {"full": [], "zoom": []}

modes["full"] = [
    (100, -400, 200),
    (100, -400, 200),
    (100, -200, 200),
    (100, 0, 500),
    (100, 0, 500),
    (100, 0, 100),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (100, -400, 400),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
    (401, -300, 300),
]
modes["zoom"] = [
    (100, -100, 100),
    (100, -100, 100),
    (100, -5, 5),
    (100, 0, 100),
    (100, 0, 100),
    (100, 0, 30),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (100, -150, 150),
    (133, -100, 100),
    (133, -100, 100),
    (133, -100, 100),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
    (133, -10, 10),
]

# Zmienne globalne
INITIAL_SIGNAL_YIELD = None
iteration_counter = 0


# --- 2. FUNKCJE POMOCNICZE ---
def extract_signal_from_file(file_path, hash):
  f_res = ROOT.TFile(file_path, "READ")
  obj_integrals = f_res.Get(f"{hash}/ChannelIntegrals")
  signal_val = 0.0
  if obj_integrals and hasattr(obj_integrals, "GetTitle"):
    title_str = obj_integrals.GetTitle()
    try:
      import json as pyjson

      data_dict = pyjson.loads(title_str)
      signal_val = float(data_dict.get("Signal", 0.0))
    except Exception as e:
      print(f"[OSTRZEŻENIE] Nie udało się sparsować ChannelIntegrals: {e}")
  f_res.Close()
  return signal_val


def evaluate_fom_at(cut1, cut2):
  """Wydzielona funkcja obliczająca FOM dla konkretnej pary cięć."""
  df_current = ROOT.RDataFrame(
      "tree_name", "/workspace/root_files/hal/local_root_cache.root"
  )
  df_data_current = ROOT.RDataFrame(
      "tree_name", "/workspace/root_files/hal/local_root_cache_data.root"
  )

  current_cut_list = list(base_cut_list)
  final_cut_string = (
      f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})"
  )
  current_cut_list.append(final_cut_string)

  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
  step_result_dir = current_result_dir
  total_cut = final_cut_string

  (vars_1D_main, titles_1D_main, modes_1D_main), (
      vars_1D_truth,
      titles_1D_truth,
      modes_1D_truth,
  ) = split_variable_group(
      variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D
  )

  hist_1D = fill_1d_histograms(
      df_current,
      df_data_current,
      channName,
      vars_1D_main,
      titles_1D_main,
      modes_1D_main,
      total_cut,
      scaling_factor,
      compute_mc_sum=True,
  )

  hist_1D_truth = {}
  if vars_1D_truth:
    truth_channels_1D = {TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]}
    hist_1D_truth = fill_1d_histograms(
        df_current,
        None,
        truth_channels_1D,
        vars_1D_truth,
        titles_1D_truth,
        modes_1D_truth,
        total_cut,
        compute_mc_sum=False,
    )

  root_file_name = f"separated_kin_fits_optimization_signal_cuts_1D_variables.root"
  hist_1D, canvas, legends = draw_and_save_1d_histograms(
      hist_1D,
      vars_1D_main,
      modes_1D_main,
      channColor,
      channTitle,
      step_result_dir,
      cut_hash,
      cuts_json_str,
      current_cut_list,
      root_file_name,
  )

  if vars_1D_truth:
    root_file_name_truth = (
        f"separated_kin_fits_optimization_1D_signal_cuts_truth_variables.root"
    )
    hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
        hist_1D_truth,
        vars_1D_truth,
        modes_1D_truth,
        channColor,
        channTitle,
        step_result_dir,
        cut_hash,
        cuts_json_str,
        current_cut_list,
        root_file_name_truth,
    )

  for canva in canvas:
    canva.Close()
  if vars_1D_truth:
    for canva in canvas_truth:
      canva.Close()

  file_path = f"{step_result_dir}/{root_file_name}"
  f_res = ROOT.TFile(file_path, "READ")
  obj_purity = f_res.Get(f"{cut_hash}/Purity")

  purity_val = 0.0
  if obj_purity and hasattr(obj_purity, "GetVal"):
    purity_val = obj_purity.GetVal()

  current_signal = extract_signal_from_file(file_path, cut_hash)
  f_res.Close()

  current_efficiency = (
      (current_signal / INITIAL_SIGNAL_YIELD)
      if INITIAL_SIGNAL_YIELD and INITIAL_SIGNAL_YIELD > 0
      else 0.0
  )

  return purity_val * current_efficiency


# --- 3. KROK POCZĄTKOWY (ITERACJA 0) - BAZOWY DLA WYDAJNOŚCI ---
print(
    "[INFO] Wykonywanie kroku początkowego (iteracja 0) bez cięcia m1_vs_m2..."
)
df_init_mc = ROOT.RDataFrame(
    "tree_name", "/workspace/root_files/hal/local_root_cache.root"
)
df_init_data = ROOT.RDataFrame(
    "tree_name", "/workspace/root_files/hal/local_root_cache_data.root"
)

init_cut_list = list(base_cut_list)
init_cuts_json_str = json.dumps(init_cut_list, indent=2)
init_cut_hash = hashlib.md5(init_cuts_json_str.encode("utf-8")).hexdigest()[:8]
init_total_cut = "true"

(vars_1D_main, titles_1D_main, modes_1D_main), (
    vars_1D_truth,
    titles_1D_truth,
    modes_1D_truth,
) = split_variable_group(
    variables_1D, variables_1D_titles, modes, TRUTH_ONLY_VARS_1D
)

hist_1D_init = fill_1d_histograms(
    df_init_mc,
    df_init_data,
    channName,
    vars_1D_main,
    titles_1D_main,
    modes_1D_main,
    init_total_cut,
    scaling_factor,
    compute_mc_sum=True,
)

root_file_name_init = "separated_kin_fits_optimization_1D_variables.root"
_, canvas_init, _ = draw_and_save_1d_histograms(
    hist_1D_init,
    vars_1D_main,
    modes_1D_main,
    channColor,
    channTitle,
    current_result_dir,
    init_cut_hash,
    init_cuts_json_str,
    init_cut_list,
    root_file_name_init,
)
for c in canvas_init:
  c.Close()

init_file_path = f"{current_result_dir}/{root_file_name_init}"
INITIAL_SIGNAL_YIELD = extract_signal_from_file(init_file_path, init_cut_hash)
print(
    "[INFO] Początkowa liczba sygnału (Initial Signal Yield):"
    f" {INITIAL_SIGNAL_YIELD:.2f}"
)


# --- 4. FUNKCJA CELU DLA NELDER-MEAD (Z ROBUSTNESS / UŚREDNIANIEM SĄSIEDZTWA) ---
def objective_function(params):
  global iteration_counter
  iteration_counter += 1

  # Rozpakowanie parametrów (obsługa 1D lub 2D)
  if len(params) == 1:
    cut1 = params[0]
    cut2 = 11.515
    is_1d = True
    print(
        f"\n[INFO] --- Nelder-Mead Iteracja {iteration_counter} --- | cut1:"
        f" {cut1:.4f} (optymalizowany), cut2 = {cut2:.4f} (stały)"
    )
  else:
    cut1, cut2 = params
    is_1d = False
    print(
        f"\n[INFO] --- Nelder-Mead Iteracja {iteration_counter} --- | cut1:"
        f" {cut1:.4f}, cut2 = {cut2:.4f}"
    )

  # Krok niepewności kalibracji / stabilności (delta)
  delta = 0.05

  # 1. Obliczamy FOM w punkcie centralnym
  fom_center = evaluate_fom_at(cut1, cut2)

  # 2. Obliczamy FOM w punktach sąsiednich (test odporności na strome zbocza / klify)
  if is_1d:
    fom_plus = evaluate_fom_at(cut1 + delta, cut2)
    fom_minus = evaluate_fom_at(cut1 - delta, cut2)
    # Średnia z punktu centralnego i sąsiadów (kara za wejście na klif, gdzie sąsiad drastycznie spada)
    robust_fom = (fom_center + fom_plus + fom_minus) / 3.0
  else:
    # Dla 2D sprawdzamy sąsiadów w obu wymiarach
    fom_c1_plus = evaluate_fom_at(cut1 + delta, cut2)
    fom_c1_minus = evaluate_fom_at(cut1 - delta, cut2)
    fom_c2_plus = evaluate_fom_at(cut1, cut2 + delta)
    fom_c2_minus = evaluate_fom_at(cut1, cut2 - delta)
    robust_fom = (
        fom_center
        + fom_c1_plus
        + fom_c1_minus
        + fom_c2_plus
        + fom_c2_minus
    ) / 5.0

  print(
      f"[INFO] FOM Centralny: {fom_center:.6f} | Robust FOM (Średnia z"
      f" otoczenia): {robust_fom:.6f}"
  )

  # Ponieważ scipy minimalizuje, zwracamy ujemny Robust FOM
  return -robust_fom


# --- 5. URUCHOMIENIE OPTYMALIZACJI ---
initial_guess = [8.086, 11.515]

print(
    "[INFO] Rozpoczynanie optymalizacji odpornej na klify (Stability-Aware"
    " Nelder-Mead)..."
)

result = minimize(
    objective_function,
    initial_guess,
    method="Nelder-Mead",
    options={
        "maxiter": 25,  # Nieco mniej iteracji ze względu na dodatkowe punkty sąsiedztwa
        "disp": True,
        "xatol": 0.05,
        "fatol": 0.0001,
    },
)

print("\n[INFO] Optymalizacja zakończona!")
print(f"Status sukcesu: {result.success}")
print(f"Komunikat: {result.message}")
if len(initial_guess) == 1:
  print(f"Optymalne cut1: {result.x[0]:.4f}")
else:
  print(f"Optymalne cut1: {result.x[0]:.4f}")
  print(f"Optymalne cut2: {result.x[1]:.4f}")
print(f"Optymalne cuty: {result.x}")
print(f"Maksymalna osiągnięta wartość Robust FOM: {-result.fun:.6f}")

[INFO] Wykonywanie kroku początkowego (iteracja 0) bez cięcia m1_vs_m2...
[INFO] Zaktualizowano wpis dla hashu 5fc561ef w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Początkowa liczba sygnału (Initial Signal Yield): 35570.53
[INFO] Rozpoczynanie optymalizacji odpornej na klify (Stability-Aware Nelder-Mead)...

[INFO] --- Nelder-Mead Iteracja 1 --- | cut1: 8.0860, cut2 = 11.5150
[INFO] Zaktualizowano wpis dla hashu be79154d w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu be79154d w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 6cc37af5 w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 6cc37af5 w rejestrze w results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 49bedf1f w rejestrze w results/s